# Análisis Predictivo de Fallas en Bomba Centrífuga Mediante Datos de Sensores

**Autor:** Diego Andrés Rivera  
**Curso:** Programación y análisis reproducible con datos  
**Lenguaje principal:** Python  
**Dataset:** `pump_sensor_data_large.csv`  
**Producto complementario:** Dashboard interactivo en Streamlit  
**Licencia:** GNU GPL v3

Este notebook desarrolla un análisis reproducible de datos de sensores asociados a una bomba centrífuga industrial. El objetivo es identificar patrones operativos relacionados con eventos de falla y evaluar modelos básicos de clasificación supervisada para apoyar decisiones de mantenimiento predictivo.


## 1. Introducción y pregunta de análisis

### 1.1 Dataset seleccionado

Para este proyecto se seleccionó el dataset **Simulated Centrifugal Pump Sensor Data for PM**, disponible en Kaggle y publicado por el usuario `naphtaly`. El archivo utilizado en este análisis es `pump_sensor_data_large.csv`.

El dataset contiene registros temporales simulados de sensores asociados a la operación de una bomba centrífuga. Las variables disponibles son:

- `timestamp`: marca temporal de cada medición.
- `vibration`: nivel de vibración del equipo.
- `temperature`: temperatura registrada.
- `flow_rate`: caudal de operación.
- `pressure`: presión del sistema.
- `power_consumption`: consumo de potencia.
- `failure`: variable binaria que indica si el registro corresponde a operación normal o a condición de falla.

Fuente del dataset: Kaggle, dataset **Simulated Centrifugal Pump Sensor Data for PM**.  
Enlace: https://www.kaggle.com/datasets/naphtaly/sensordatarandomgenerated  
Archivo utilizado: `pump_sensor_data_large.csv`.  
Fecha de descarga: 6 de mayo de 2026.  
Licencia: **Community Data License Agreement – Sharing – Version 1.0**.

### 1.2 Pregunta de análisis

La pregunta principal del proyecto es:

**¿Qué patrones operativos permiten identificar eventos de falla en una bomba centrífuga y qué tan efectivo es un modelo de clasificación supervisada para detectar dichas condiciones a partir de datos de sensores?**

Esta pregunta es verificable con los datos porque el dataset contiene una variable objetivo binaria (`failure`), que permite diferenciar registros de operación normal y registros asociados a falla. Además, las variables de sensores permiten comparar el comportamiento operativo de la bomba bajo ambas condiciones.

### 1.3 Relevancia del problema

El análisis es relevante porque las bombas centrífugas son equipos críticos en sistemas industriales de transporte de fluidos. Una condición de falla no detectada puede generar paradas no programadas, reducción de disponibilidad, afectación de procesos y aumento de costos de mantenimiento.

El uso de datos de sensores permite explorar una aproximación inicial al mantenimiento predictivo. Variables como vibración, temperatura, caudal, presión y consumo de potencia pueden aportar información útil para diferenciar condiciones normales de condiciones anómalas.

Aunque el dataset es simulado, resulta adecuado para desarrollar un flujo reproducible de análisis de datos aplicado a un caso industrial. Además, permite integrar carga de datos, exploración inicial, limpieza, transformación, feature engineering, visualizaciones, clasificación supervisada y evaluación de modelos, que son componentes centrales del proyecto final.


## 2. Carga y exploración inicial

En esta sección se preparan las librerías necesarias, se carga el dataset desde el repositorio de GitHub y se realiza una primera exploración de su estructura. El objetivo es verificar que los datos puedan importarse de forma reproducible y describir sus características iniciales antes de aplicar limpieza, transformación o modelado.

### 2.1 Carga de librerías y configuración del entorno

In [24]:
# Librerías para manejo de datos
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Librerías para estadística y modelado
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Configuración general
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

RANDOM_STATE = 42

Las librerías importadas permiten realizar carga de datos, exploración inicial, visualización, análisis estadístico y modelado supervisado. Se define además una semilla aleatoria (`RANDOM_STATE = 42`) para favorecer la reproducibilidad de los resultados.

### 2.2 Carga reproducible del dataset

El notebook fue abierto desde GitHub en Google Colab. Para garantizar que el dataset esté disponible en el entorno de ejecución, se clona el repositorio del proyecto y se carga el archivo `pump_sensor_data_large.csv` desde la carpeta `data/`.

In [25]:
# ==========================
# CONFIGURACIÓN DEL REPOSITORIO
# ==========================

REPO_URL = "https://github.com/diego-rivera-eng/predictive-maintenance-pump-sensors.git"
REPO_NAME = "predictive-maintenance-pump-sensors"
DATA_FILE = "pump_sensor_data_large.csv"

# Ruta donde se clonará el repositorio en Colab
repo_path = Path("/content") / REPO_NAME

# Clonar el repositorio si aún no existe
if not repo_path.exists():
    print("Clonando repositorio desde GitHub...")
    os.system(f"git clone {REPO_URL} {repo_path}")
else:
    print("El repositorio ya existe en Colab. Actualizando...")
    os.system(f"cd {repo_path} && git pull")

# Ruta absoluta del dataset dentro del repositorio clonado
data_path = repo_path / "data" / DATA_FILE

# Verificar existencia del archivo
if not data_path.exists():
    raise FileNotFoundError(f"No se encontró el dataset en: {data_path}")

# Cargar dataset
df_raw = pd.read_csv(data_path)

print("Dataset cargado correctamente.")
print(f"Ruta usada: {data_path}")
print(f"Dimensiones: {df_raw.shape}")

El repositorio ya existe en Colab. Actualizando...
Dataset cargado correctamente.
Ruta usada: /content/predictive-maintenance-pump-sensors/data/pump_sensor_data_large.csv
Dimensiones: (100000, 7)


### 2.3 Vista inicial del dataset

In [26]:
# Vista inicial del dataset
df_raw.head()

,timestamp,vibration,temperature,flow_rate,pressure,power_consumption,failure
0,2024-01-01 00:00:00,5.4967,65.1530,378.0920,7.2004,6.5177,0
1,2024-01-01 00:01:00,4.8617,54.2232,295.2886,11.0956,4.9260,0
2,2024-01-01 00:02:00,13.5354,73.1682,233.5232,9.7585,6.3383,1
3,2024-01-01 00:03:00,6.5230,56.9038,230.5681,9.4335,5.2575,0
4,2024-01-01 00:04:00,4.7658,58.3630,282.8675,11.5351,4.9681,0


### 2.4 Estructura general: dimensiones y tipos de variables

In [27]:
# Estructura general del dataset
print("Información general del dataset:")
print(f"Filas: {df_raw.shape[0]}")
print(f"Columnas: {df_raw.shape[1]}")

df_raw.info()

Información general del dataset:
Filas: 100000
Columnas: 7
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   timestamp          100000 non-null  object 
 1   vibration          100000 non-null  float64
 2   temperature        100000 non-null  float64
 3   flow_rate          100000 non-null  float64
 4   pressure           100000 non-null  float64
 5   power_consumption  100000 non-null  float64
 6   failure            100000 non-null  int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 5.3+ MB


### 2.5 Valores únicos por columna

In [28]:
# Resumen de columnas, tipos de dato y valores únicos
estructura_df = pd.DataFrame({
    "columna": df_raw.columns,
    "tipo_dato": df_raw.dtypes.astype(str).values,
    "valores_no_nulos": df_raw.notna().sum().values,
    "valores_unicos": df_raw.nunique().values
})

estructura_df

,columna,tipo_dato,valores_no_nulos,valores_unicos
0,timestamp,object,100000,100000
1,vibration,float64,100000,100000
2,temperature,float64,100000,100000
3,flow_rate,float64,100000,100000
4,pressure,float64,100000,100000
5,power_consumption,float64,100000,100000
6,failure,int64,100000,2


### 2.6 Estadísticas descriptivas


In [29]:
# Estadísticas descriptivas iniciales de las variables numéricas
df_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
vibration,100000.0000,5.1986,1.7562,0.5344,4.3410,5.0279,5.7272,22.1252
temperature,100000.0000,60.3056,5.4339,38.1298,56.7096,60.1348,63.6071,93.5080
flow_rate,100000.0000,299.9246,49.9926,79.3057,266.2901,299.8640,333.6022,510.9683
pressure,100000.0000,9.9997,2.0032,1.3741,8.6418,10.0052,11.3453,17.8364
power_consumption,100000.0000,5.0373,0.5800,2.5853,4.6665,5.0089,5.3608,9.2528
failure,100000.0000,0.0200,0.1400,0.0000,0.0000,0.0000,0.0000,1.0000


### 2.7 Valores faltantes y duplicados

In [30]:
# Valores faltantes por columna
missing_summary = pd.DataFrame({
    "n_missing": df_raw.isna().sum(),
    "pct_missing": df_raw.isna().mean() * 100
})

missing_summary

,n_missing,pct_missing
timestamp,0,0.0000
vibration,0,0.0000
temperature,0,0.0000
flow_rate,0,0.0000
pressure,0,0.0000
power_consumption,0,0.0000
failure,0,0.0000


In [31]:
# Duplicados
duplicados = df_raw.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

Registros duplicados: 0


### 2.8 Distribución inicial de la variable objetivo

In [32]:
# Distribución incial de la variable objetivo
target_distribution = (
    df_raw["failure"]
    .value_counts()
    .to_frame("n_registros")
)

target_distribution["porcentaje"] = (
    target_distribution["n_registros"] / len(df_raw) * 100
).round(4)

target_distribution

,n_registros,porcentaje
failure,,
0,98000,98.0000
1,2000,2.0000


### Primeras observaciones

El dataset se cargó correctamente desde el repositorio del proyecto. Contiene 100.000 registros y 7 columnas: una variable temporal (`timestamp`), cinco variables numéricas de sensores y una variable objetivo binaria (`failure`).

La columna `timestamp` aparece inicialmente como tipo `object`, por lo que deberá convertirse a formato fecha/hora en la etapa de limpieza y transformación. Las variables de sensores son numéricas continuas y no presentan valores faltantes en esta exploración inicial.

No se identifican registros duplicados. La variable objetivo `failure` contiene dos clases: operación normal (`0`) y condición de falla (`1`). La clase de falla representa el 2 % de los registros, por lo que el problema presenta desbalance de clases. Este aspecto será considerado posteriormente en la selección de métricas de evaluación del modelo.

## 5. Inspección de calidad de datos

Se revisan valores faltantes, duplicados, tipos de datos, valores únicos y distribución de la variable objetivo `failure`. Aunque el dataset parezca limpio, esta revisión debe quedar documentada para cumplir con el criterio de reproducibilidad y trazabilidad del análisis.


In [ ]:

missing_summary = df_raw.isna().sum().to_frame("n_missing")
missing_summary["pct_missing"] = 100 * missing_summary["n_missing"] / len(df_raw)
missing_summary


In [ ]:

duplicados = df_raw.duplicated().sum()
print(f"Registros duplicados: {duplicados:,}")


In [ ]:

unique_summary = df_raw.nunique().to_frame("n_unique")
unique_summary


In [ ]:

target_counts = df_raw["failure"].value_counts().sort_index()
target_pct = df_raw["failure"].value_counts(normalize=True).sort_index() * 100

balance_df = pd.DataFrame({
    "n_registros": target_counts,
    "porcentaje": target_pct
})

balance_df


### Interpretación

La inspección de calidad muestra que el dataset no presenta valores faltantes ni registros duplicados. Por tanto, en esta etapa no se requiere imputación, eliminación de registros incompletos ni depuración por duplicidad.

La variable objetivo `failure` está correctamente codificada como variable binaria, con los valores `0` y `1`. Sin embargo, su distribución evidencia un desbalance importante: el 98 % de los registros corresponde a operación normal y el 2 % a condición de falla.

Este desbalance será relevante en la etapa de modelado, porque una alta exactitud global podría no representar una buena capacidad de detección de fallas. Por esta razón, además de `accuracy`, se evaluarán métricas como `precision`, `recall`, `F1-score` y matriz de confusión.

## 6. Limpieza y transformación inicial

En esta sección se realizan las transformaciones mínimas necesarias para preparar el dataset antes del análisis exploratorio y del modelado. Se convierte la columna temporal, se ordenan los registros cronológicamente, se valida la codificación de la variable objetivo y se revisan rangos físicos básicos de las señales de sensores.


In [ ]:
df = df_raw.copy()

# Conversión temporal
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Orden cronológico
df = df.sort_values("timestamp").reset_index(drop=True)

# Validaciones básicas
print("Tipo de timestamp:", df["timestamp"].dtype)
print("Valores únicos en failure:", sorted(df["failure"].unique()))
print("Fechas:", df["timestamp"].min(), "a", df["timestamp"].max())

df.head()


In [ ]:
# Revisión de rangos físicos o plausibles
sensor_cols = ["vibration", "temperature", "flow_rate", "pressure", "power_consumption"]

range_summary = df[sensor_cols].agg(["min", "max", "mean", "std"]).T
range_summary


### Interpretación

La columna `timestamp` fue convertida correctamente a formato `datetime64[ns]`, lo que permite realizar análisis temporal, ordenar cronológicamente los registros y construir variables derivadas basadas en tiempo.

Después de ordenar el dataset, el periodo cubierto por los datos va desde el **1 de enero de 2024 a las 00:00:00** hasta el **10 de marzo de 2024 a las 10:39:00**. La secuencia temporal queda preparada para análisis de series de tiempo y para la construcción posterior de variables como diferencias entre mediciones, medias móviles y ventanas de comportamiento.

La variable objetivo `failure` conserva únicamente los valores `0` y `1`, por lo que se mantiene correctamente definida como variable binaria para clasificación.

La revisión de rangos físicos muestra que las variables de sensores presentan valores positivos y plausibles para un caso de monitoreo operativo. No se observan valores negativos en caudal, presión, potencia, temperatura o vibración, por lo que no se requiere eliminación inmediata por inconsistencias físicas básicas.

Los valores máximos de vibración y temperatura son considerablemente superiores a sus valores promedio. Estos registros no se eliminan, ya que pueden contener información relevante asociada a condiciones anómalas o eventos de falla.

## 7. Feature engineering

Se crean variables derivadas para representar comportamiento temporal, cambios entre mediciones, relaciones operativas y tendencias móviles. Estas variables buscan aportar información adicional para detectar condiciones anómalas.


In [ ]:
# Variables temporales
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["elapsed_minutes"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds() / 60

# Representación cíclica de la hora
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Diferencias entre registros consecutivos
for col in sensor_cols:
    df[f"{col}_diff"] = df[col].diff().fillna(0)

# Ratios operativos. Se agrega epsilon para evitar división por cero.
eps = 1e-6
df["power_flow_ratio"] = df["power_consumption"] / (df["flow_rate"] + eps)
df["pressure_flow_ratio"] = df["pressure"] / (df["flow_rate"] + eps)
df["vibration_power_ratio"] = df["vibration"] / (df["power_consumption"] + eps)

# Variables móviles con ventana de 10 minutos
window = 10
for col in ["vibration", "temperature", "pressure", "power_consumption"]:
    df[f"{col}_roll_mean_{window}"] = df[col].rolling(window=window, min_periods=1).mean()
    df[f"{col}_roll_std_{window}"] = df[col].rolling(window=window, min_periods=1).std().fillna(0)

print(f"Columnas después de feature engineering: {df.shape[1]}")
df.head()


In [ ]:
# Lista final de variables derivadas
engineered_cols = [col for col in df.columns if col not in df_raw.columns]
engineered_cols


**Interpretación esperada:**  
Las variables instantáneas describen el estado de la bomba en un momento específico. Las variables derivadas incorporan tendencia, variabilidad y relaciones operativas que pueden mejorar la detección de fallas frente al uso exclusivo de señales instantáneas.


## 8. Análisis exploratorio de datos

Esta sección debe incluir al menos cinco visualizaciones relevantes, cada una con título, etiquetas e interpretación. El objetivo es conectar los gráficos con la pregunta de análisis, no solo mostrar figuras.


### 8.1 Balance de clases

Este gráfico muestra la proporción entre registros normales y registros de falla.


In [ ]:
fig = px.bar(
    balance_df.reset_index().rename(columns={"index": "failure"}),
    x="failure",
    y="n_registros",
    text="porcentaje",
    title="Distribución de clases: operación normal vs falla",
    labels={"failure": "Clase failure", "n_registros": "Número de registros"}
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()


**Interpretación pendiente:**  
Describir la proporción de fallas y explicar por qué esto afecta la selección de métricas de evaluación.


### 8.2 Serie temporal interactiva de sensores

Este gráfico permite observar la evolución temporal de las señales principales y ubicar visualmente eventos de falla.


In [ ]:
sensor_to_plot = "vibration"  # Cambiar por: temperature, flow_rate, pressure, power_consumption

fig = px.line(
    df,
    x="timestamp",
    y=sensor_to_plot,
    color=df["failure"].astype(str),
    title=f"Serie temporal de {sensor_to_plot} coloreada por condición de falla",
    labels={"timestamp": "Tiempo", sensor_to_plot: sensor_to_plot, "color": "failure"}
)
fig.show()


**Interpretación pendiente:**  
Indicar si los eventos de falla se concentran en ciertos niveles del sensor seleccionado o si aparecen distribuidos aleatoriamente en el tiempo.


### 8.3 Comparación de sensores por clase

Los boxplots permiten comparar el comportamiento de cada sensor en operación normal y en falla.


In [ ]:
df_long = df.melt(
    id_vars=["failure"],
    value_vars=sensor_cols,
    var_name="sensor",
    value_name="value"
)

fig = px.box(
    df_long,
    x="sensor",
    y="value",
    color=df_long["failure"].astype(str),
    title="Comparación de sensores por condición normal/falla",
    labels={"sensor": "Sensor", "value": "Valor", "color": "failure"}
)
fig.show()


**Interpretación pendiente:**  
Identificar qué sensores muestran mayor separación entre registros normales y fallas.


### 8.4 Distribución de sensores clave

Se comparan histogramas por clase para sensores seleccionados.


In [ ]:
sensor_to_hist = "vibration"  # Cambiar según interés

fig = px.histogram(
    df,
    x=sensor_to_hist,
    color=df["failure"].astype(str),
    nbins=60,
    marginal="box",
    opacity=0.70,
    title=f"Distribución de {sensor_to_hist} por clase",
    labels={sensor_to_hist: sensor_to_hist, "color": "failure"}
)
fig.show()


**Interpretación pendiente:**  
Describir si existe solapamiento entre clases o si la variable separa claramente condiciones normales y fallas.


### 8.5 Matriz de correlación

La matriz de correlación permite observar relaciones lineales entre sensores, variables derivadas y la variable objetivo.


In [ ]:
corr_cols = sensor_cols + [
    "power_flow_ratio",
    "pressure_flow_ratio",
    "vibration_power_ratio",
    "vibration_roll_mean_10",
    "temperature_roll_mean_10",
    "failure"
]

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación entre sensores, variables derivadas y falla")
plt.tight_layout()
plt.show()


**Interpretación pendiente:**  
Señalar qué variables tienen mayor asociación lineal con `failure`. Recordar que correlación no implica causalidad.


### 8.6 PCA 2D para visualización multivariable

Se utiliza PCA para proyectar las variables numéricas en dos componentes y observar si existe separación visual entre clases.


In [ ]:
feature_cols_for_pca = sensor_cols + engineered_cols

X_pca = df[feature_cols_for_pca].replace([np.inf, -np.inf], np.nan).fillna(0)

scaler_pca = StandardScaler()
X_scaled_pca = scaler_pca.fit_transform(X_pca)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_components = pca.fit_transform(X_scaled_pca)

pca_df = pd.DataFrame({
    "PC1": pca_components[:, 0],
    "PC2": pca_components[:, 1],
    "failure": df["failure"].astype(str)
})

fig = px.scatter(
    pca_df.sample(n=min(10000, len(pca_df)), random_state=RANDOM_STATE),
    x="PC1",
    y="PC2",
    color="failure",
    title="Proyección PCA 2D de sensores y variables derivadas",
    labels={"failure": "failure"}
)
fig.show()

print("Varianza explicada por PC1 y PC2:", pca.explained_variance_ratio_)


**Interpretación pendiente:**  
Comentar si las clases aparecen mezcladas o separadas en el espacio reducido. Si hay separación fuerte, discutir si el dataset puede ser sintético o altamente separable.


## 9. Análisis estadístico

Se aplica una prueba estadística para evaluar si existen diferencias significativas entre registros normales y registros de falla en sensores clave. Esta prueba no demuestra causalidad, pero respalda cuantitativamente las diferencias observadas en el EDA.


In [ ]:
def mann_whitney_test_by_failure(data, variable, target="failure"):
    normal = data.loc[data[target] == 0, variable]
    failure = data.loc[data[target] == 1, variable]

    stat, p_value = stats.mannwhitneyu(normal, failure, alternative="two-sided")

    return {
        "variable": variable,
        "normal_median": normal.median(),
        "failure_median": failure.median(),
        "statistic": stat,
        "p_value": p_value
    }

test_results = pd.DataFrame([
    mann_whitney_test_by_failure(df, col) for col in sensor_cols
])

test_results


**Interpretación pendiente:**  
Si el p-valor es menor que 0.05, se rechaza la hipótesis nula de igualdad de distribuciones entre operación normal y falla para esa variable. Debe interpretarse junto con la magnitud de la diferencia y no solo con significancia estadística.


## 10. Preparación para modelado

Se define la matriz de variables predictoras `X` y la variable objetivo `y`. La división train/test se realiza de forma estratificada para preservar la proporción de fallas en ambos conjuntos.


In [ ]:
# Selección de variables predictoras
feature_cols = sensor_cols + engineered_cols

# Limpieza final de infinitos o NaN generados por ratios
model_df = df[feature_cols + ["failure"]].replace([np.inf, -np.inf], np.nan).fillna(0)

X = model_df[feature_cols]
y = model_df["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Distribución y_train:")
print(y_train.value_counts(normalize=True))
print("Distribución y_test:")
print(y_test.value_counts(normalize=True))


## 11. Modelo baseline

El modelo baseline predice siempre la clase mayoritaria. Sirve para mostrar que, en un problema desbalanceado, una alta exactitud puede ser engañosa si el modelo no detecta fallas.


In [ ]:
def evaluate_classifier(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_score)
        pr_auc = average_precision_score(y_test, y_score)
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    metrics = {
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_falla": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_falla": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_falla": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

    return metrics, y_pred

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_metrics, baseline_pred = evaluate_classifier("Baseline clase mayoritaria", baseline, X_test, y_test)
baseline_metrics


In [ ]:
print(classification_report(y_test, baseline_pred, zero_division=0))


**Interpretación pendiente:**  
Explicar por qué el modelo baseline puede tener accuracy alta, pero recall de falla igual a cero o muy bajo.


## 12. Regresión logística

La regresión logística se utiliza como modelo interpretable de clasificación binaria. Permite estimar la probabilidad de falla a partir de sensores y variables derivadas.


In [ ]:
logistic_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight="balanced"
    ))
])

logistic_model.fit(X_train, y_train)

logistic_metrics, logistic_pred = evaluate_classifier("Regresión logística", logistic_model, X_test, y_test)
logistic_metrics


In [ ]:
print(classification_report(y_test, logistic_pred, zero_division=0))

cm = confusion_matrix(y_test, logistic_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Falla"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusión - Regresión logística")
plt.show()


In [ ]:
# Coeficientes de regresión logística
coef = logistic_model.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({
    "variable": feature_cols,
    "coeficiente": coef,
    "abs_coeficiente": np.abs(coef)
}).sort_values("abs_coeficiente", ascending=False)

coef_df.head(15)


**Interpretación pendiente:**  
Analizar las variables con mayor magnitud de coeficiente. Recordar que el signo indica dirección de asociación con la probabilidad de falla en el espacio escalado, no causalidad física definitiva.


## 13. Modelo comparativo: Random Forest

Se entrena un Random Forest como modelo no lineal de comparación. También permite obtener una medida de importancia de variables.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_metrics, rf_pred = evaluate_classifier("Random Forest", rf_model, X_test, y_test)
rf_metrics


In [ ]:
print(classification_report(y_test, rf_pred, zero_division=0))

cm = confusion_matrix(y_test, rf_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Falla"])
disp.plot(cmap="Greens")
plt.title("Matriz de confusión - Random Forest")
plt.show()


In [ ]:
importance_df = pd.DataFrame({
    "variable": feature_cols,
    "importancia": rf_model.feature_importances_
}).sort_values("importancia", ascending=False)

importance_df.head(15)


**Interpretación pendiente:**  
Comparar si las variables más importantes coinciden con los hallazgos del EDA y con los coeficientes de la regresión logística.


## 14. Comparación de modelos

Se comparan los modelos evaluados usando métricas adecuadas para clasificación desbalanceada.


In [ ]:
metrics_df = pd.DataFrame([
    baseline_metrics,
    logistic_metrics,
    rf_metrics
])

metrics_df


In [ ]:
fig = px.bar(
    metrics_df.melt(id_vars="modelo", var_name="metrica", value_name="valor"),
    x="metrica",
    y="valor",
    color="modelo",
    barmode="group",
    title="Comparación de métricas de clasificación por modelo",
    labels={"metrica": "Métrica", "valor": "Valor", "modelo": "Modelo"}
)
fig.show()


**Interpretación pendiente:**  
Indicar cuál modelo es más conveniente para el problema y justificarlo. En mantenimiento predictivo, el recall de la clase falla suele ser crítico porque representa la capacidad de detectar fallas reales.


## 15. Dashboard en Streamlit

Como complemento al notebook, se desarrollará una aplicación en Streamlit que permita explorar los datos de forma interactiva.

**Módulos esperados de la app:**

1. Resumen del dataset.
2. Exploración temporal de sensores.
3. Comparación entre operación normal y falla.
4. Variables derivadas de feature engineering.
5. Modelado y métricas.
6. Conclusiones principales.

**Pendiente:** incluir enlace de la app desplegada una vez esté publicada.


## 16. Uso documentado de IA

La siguiente tabla documenta cómo se usaron herramientas de IA durante el proyecto. La idea no es ocultar el uso de IA, sino demostrar criterio propio al revisar, corregir y adaptar sus respuestas.

| Herramienta | Qué se pidió | Qué entregó | Qué se ajustó | Validación propia |
|---|---|---|---|---|
| ChatGPT | Estructura del proyecto final | Plan de trabajo y secciones | Se adaptó al dataset de sensores | Se contrastó con la rúbrica del curso |
| ChatGPT | Ideas de feature engineering | Variables temporales, ratios y medias móviles | Se seleccionaron variables interpretables | Se verificó que fueran calculables con el dataset |
| ChatGPT/Copilot | Apoyo para código de gráficos/modelado | Fragmentos de código base | Se corrigieron nombres de columnas y rutas | Se ejecutó el notebook de principio a fin |
| ChatGPT | Interpretación de métricas | Texto explicativo | Se ajustó para evitar sobreinterpretación | Se contrastó con matriz de confusión y métricas reales |


## 17. Conclusiones

### Respuesta a la pregunta principal

**Pendiente de completar después del análisis final.**

Se espera responder de forma directa:

- Qué patrones operativos permiten identificar fallas.
- Qué sensores resultan más relevantes.
- Qué tan bien funcionan los modelos evaluados.
- Qué limitaciones tiene el análisis.

### Limitaciones

1. El dataset se interpreta como un caso académico o demostrativo si no se dispone de procedencia industrial completa.
2. El análisis detecta asociaciones entre sensores y falla, pero no prueba causalidad física.
3. No se modela tiempo restante hasta falla.
4. No se validan umbrales contra normas, fabricante o historial real de mantenimiento.
5. Los modelos requieren validación con datos reales adicionales antes de uso operacional.

### Recomendación concreta

**Pendiente:** formular una recomendación basada en los hallazgos finales. Por ejemplo, priorizar monitoreo de sensores con mayor asociación a falla, complementar con ventanas móviles y evaluar el modelo con énfasis en recall de falla.


## 18. Checklist final antes de entregar

- [ ] El notebook corre de principio a fin sin errores.
- [ ] La pregunta de análisis está claramente formulada.
- [ ] La limpieza está documentada con justificación.
- [ ] Hay al menos cinco visualizaciones con título, etiquetas e interpretación.
- [ ] Hay al menos una prueba estadística o un modelo con interpretación.
- [ ] Las conclusiones responden la pregunta principal.
- [ ] El README explica el proyecto y cómo reproducirlo.
- [ ] Existe `requirements.txt`.
- [ ] El repositorio tiene al menos tres commits significativos.
- [ ] La app Streamlit funciona y tiene gráficos interactivos.
- [ ] El uso de IA está documentado críticamente.
